In [1]:
from Bio import Entrez
import os
import requests
import json

In [2]:
Entrez.email = "hr75155@gmail.com"

In [3]:
def get_pmcid_from_pmid(pmids):
    try:
        base_url = "https://pmc.ncbi.nlm.nih.gov/tools/idconv/api/v1/articles/"
        response = requests.get(base_url, params={"tool": "alzheimers-rag", "email": "hr75155@gmail.com", "ids": ",".join(pmids), "format": "json"})
        response = response.json()
        pmcid_list = [record["pmcid"] for record in response.get("records", []) if "pmcid" in record]
        return pmcid_list
    except Exception as e:
        print(f"Error fetching PMC IDs: {e}")
        return []

In [2]:
def download_pmc_papers(pmc_ids, output_dir="../data/raw/", file_name="pmc_corpus.jsonl"):
    os.makedirs(output_dir, exist_ok=True)
    base_url = "https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi"
    save_path = os.path.join(output_dir, file_name)
    
    for pmcid in pmc_ids:
        clean_id = str(pmcid).upper().replace("PMC", "")
        full_pmcid = f"PMC{clean_id}"
        
        # We enforce json here because we want to parse it cleanly first
        file_url = f"{base_url}/BioC_json/{full_pmcid}/unicode"
        
        print(f"Fetching {full_pmcid}...")
        
        try:
            response = requests.get(file_url, timeout=15)
            
            if response.status_code == 200:
                # Parse to ensure it's valid JSON
                paper_data = response.json() 
                
                # Append to the JSONL file: single line + newline character
                with open(save_path, "a", encoding="utf-8") as f:
                    f.write(json.dumps(paper_data) + "\n")
                    
                print(f"  Successfully appended to {save_path}")
                
            elif response.status_code in [403, 404]:
                print(f"  [Skipped] {full_pmcid} not in Open Access subset or path differs. Status: {response.status_code}")
            else:
                print(f"  [Failed] HTTP Status {response.status_code}")
                
        except Exception as e:
            print(f"  [Error] Could not process {full_pmcid}: {e}")

In [5]:
handle = Entrez.esearch(
    db="pubmed",
    term="Alzheimer Disease AND (amyloid OR tau OR biomarker OR treatment)",
    retmax=3000,
)

record = Entrez.read(handle)

pmids = record["IdList"]

In [6]:
len(pmids)

3000

In [7]:
pmcids = []
for pmid_batch in [pmids[i:i + 100] for i in range(0, len(pmids), 100)]:
    pmcid_response = get_pmcid_from_pmid(pmid_batch)
    pmcids.extend(pmcid_response)

In [8]:
download_pmc_papers(pmcids)

Fetching PMC13379251...
  [Error] Could not process PMC13379251: Expecting value: line 1 column 2 (char 1)
Fetching PMC13370628...
  Successfully appended to ../data/raw/pmc_corpus.jsonl
Fetching PMC13373120...
  Successfully appended to ../data/raw/pmc_corpus.jsonl
Fetching PMC13370888...
  Successfully appended to ../data/raw/pmc_corpus.jsonl
Fetching PMC13370575...
  Successfully appended to ../data/raw/pmc_corpus.jsonl
Fetching PMC13370547...
  Successfully appended to ../data/raw/pmc_corpus.jsonl
Fetching PMC13372720...
  Successfully appended to ../data/raw/pmc_corpus.jsonl
Fetching PMC13373788...
  Successfully appended to ../data/raw/pmc_corpus.jsonl
Fetching PMC13372883...
  Successfully appended to ../data/raw/pmc_corpus.jsonl
Fetching PMC13372599...
  Successfully appended to ../data/raw/pmc_corpus.jsonl
Fetching PMC13372759...
  Successfully appended to ../data/raw/pmc_corpus.jsonl
Fetching PMC13370964...
  Successfully appended to ../data/raw/pmc_corpus.jsonl
Fetching PMC1